# Pipeline Preprocessing Log Chat WhatsApp — Kantin Baiiq

Dokumentasi arsitektur pemrosesan data untuk mentransformasikan log teks tidak terstruktur hasil ekspor WhatsApp (`chat_mentah.txt`) menjadi dataset tabular siap analisis (`dataset_wa.csv`).

### Ringkasan Alur Pemrosesan Data:
- **Ekstraksi Token**: Penguraian stempel waktu, identitas pengirim, dan isi pesan menggunakan ekspresi reguler deterministik.
- **Rekonstruksi Multiline**: Penggabungan kembali baris percakapan lanjutan yang terpisah oleh karakter enter (`\n`).
- **Penyaringan Event Sistem**: Eliminasi log sistem internal (enkripsi end-to-end, notifikasi perubahan anggota grup).
- **Masking & Pseudonimisasi PII**: Penerapan prinsip perlindungan privasi data pribadi pelanggan dengan pemetaan ID deterministik.
- **Persistensi Tabular**: Ekspor dataset terstruktur ke format CSV terstandarisasi.


## Inisialisasi Modul dan Konfigurasi I/O

Modul eksternal yang diinisialisasi:
- `re`: Ekstraksi pola string teks melalui *regular expression engine*.
- `pandas`: Struktur data tabel (*DataFrame*), transformasi kolom, dan I/O berkas tabular.

Definisi jalur berkas:
- `file_input`: Berkas sumber data teks mentah ekspor WhatsApp.
- `file_output`: Berkas target dataset hasil kurasi dan pembersihan.


In [5]:
import re
import pandas as pd

# Konfigurasi path data input dan output
file_input = 'chat_mentah.txt'
file_output = 'dataset_wa.csv'

## Spesifikasi Pola Ekstraksi Regular Expression

Struktur baku baris pesan WhatsApp ekspor teks:
`[Tanggal], [Waktu] - [Pengirim]: [Pesan]`

Komponen ekspresi reguler `pola_wa`:
- `^(\d{1,2}/\d{1,2}/\d{2,4})`: Memvalidasi posisi awal baris (`^`) dan menangkap tanggal format numerik (1–2 digit hari/bulan, 2–4 digit tahun).
- `, (\d{1,2}:\d{2}\s?[AP]M)`: Menangkap jam dan menit dengan penanda AM/PM. Karakter `\s?` mengakomodasi spasi standar maupun karakter Unicode spasi tipis (*Narrow No-Break Space* `\u202f`).
- ` - (.*?): `: Menangkap identitas pengirim hingga kemunculan token titik dua pertama sebagai pemisah pesan.
- ` (.*)`: Menangkap payload teks pesan.


In [10]:
# Definisi pola ekspresi reguler untuk menguraikan baris obrolan WhatsApp
pola_wa = r'^(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}\s?[AP]M) - (.*?): (.*)'

## Parser dan Penanganan Multiline (`baca_file_wa`)

Fungsi membaca berkas secara sekuensial per baris (*streaming execution*) guna meminimalkan penggunaan memori pada berkas log berukuran besar, dengan aturan evaluasi kondisi:

- **Kondisi 1 — Pesan Baru**: Jika baris tervalidasi oleh `pola_wa`, pesan aktif sebelumnya dikomit ke list data utama, kemudian state diperbarui dengan atribut pesan baru.
- **Kondisi 2 — Log Sistem WhatsApp**: Jika baris memuat stempel waktu tanpa pemisah pengirim (`:`), seperti notifikasi enkripsi atau mutasi anggota grup, baris tersebut difilter melalui `pola_sistem`. State sebelumnya dikomit dan buffer aktif direset agar log sistem tidak mencemari percakapan pengguna.
- **Kondisi 3 — Teks Lanjutan (Multiline)**: Jika baris tidak memiliki stempel waktu, baris diklasifikasikan sebagai kelanjutan pesan sebelumnya (enter) dan digabungkan menggunakan pembatas `\n`.


In [11]:
def baca_file_wa(chat_mentah):
    """
    Mem-parsing berkas ekspor chat WhatsApp dan merekonstruksi pesan multiline.
    
    Parameter:
        chat_mentah (str): Path berkas log mentah WhatsApp.
    
    Returns:
        list: Koleksi record tabular [Tanggal, Waktu, Pengirim, Pesan].
    """
    semua_data = []
    
    # State penyimpan atribut pesan aktif saat pembacaan sekuensial
    tgl_sementara = ""
    waktu_sementara = ""
    pengirim_sementara = ""
    pesan_sementara = ""
    
    # Pola deteksi pesan sistem WhatsApp (memiliki stempel waktu tanpa pemisah pengirim ':')
    pola_sistem = r'^(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}\s?[AP]M) - (.*)'
    
    with open(chat_mentah, 'r', encoding='utf-8') as file_txt:
        for baris in file_txt:
            baris = baris.strip()
            if not baris:
                continue
            
            pencocokan = re.match(pola_wa, baris)
            
            if pencocokan:
                # Commit pesan sebelumnya ke dataset jika ada record aktif yang valid
                if pengirim_sementara != "":
                    semua_data.append([tgl_sementara, waktu_sementara, pengirim_sementara, pesan_sementara])
                
                # Inisialisasi record baru dari grup regex
                tgl_sementara = pencocokan.group(1)
                waktu_sementara = pencocokan.group(2)
                pengirim_sementara = pencocokan.group(3)
                pesan_sementara = pencocokan.group(4)
                
            elif re.match(pola_sistem, baris):
                # Commit record aktif sebelumnya, kemudian reset state untuk mengabaikan log sistem
                if pengirim_sementara != "":
                    semua_data.append([tgl_sementara, waktu_sementara, pengirim_sementara, pesan_sementara])
                tgl_sementara = ""
                waktu_sementara = ""
                pengirim_sementara = ""
                pesan_sementara = ""
                
            else:
                # Rekonstruksi pesan multiline (tambahkan baris lanjutan ke pesan aktif)
                if pengirim_sementara != "":
                    pesan_sementara = pesan_sementara + "\n" + baris
        
        # Commit record terakhir setelah pembacaan stream berkas selesai
        if pengirim_sementara != "":
            semua_data.append([tgl_sementara, waktu_sementara, pengirim_sementara, pesan_sementara])
            
    return semua_data

## Eksekusi Ekstraksi dan Pembentukan DataFrame

Konversi array record hasil ekstraksi menjadi struktur data `pandas.DataFrame` dengan skema kolom:
- `Tanggal`: Tanggal transmisi pesan.
- `Waktu`: Waktu transmisi pesan beserta indikator AM/PM.
- `Pengirim`: Identitas asal transmisi (nama kontak atau nomor kontak).
- `Pesan`: Konten pesan lengkap pasca-rekonstruksi multiline.


In [12]:
# Ekstraksi data mentah ke list record
print("Memulai proses membaca dan membersihkan file WhatsApp...")
hasil_data = baca_file_wa(file_input)

# Instansiasi DataFrame dengan struktur kolom standar
df = pd.DataFrame(hasil_data, columns=['Tanggal', 'Waktu', 'Pengirim', 'Pesan'])

# Validasi sampel data awal
display(df.head())

Memulai proses membaca dan membersihkan file WhatsApp...


,Tanggal,Waktu,Pengirim,Pesan
0,7/27/25,12:02 PM,Afri Kantin Baiiq MBR,<Media omitted>
1,7/27/25,5:06 PM,+62 821-1146-0799,nuhun infona bu 🙏🙏
2,7/27/25,5:07 PM,+62 816-4801-781,Walaikumsalam wrwb. Nasi tumpeng merah putih y...
3,7/27/25,5:12 PM,Afri Kantin Baiiq MBR,samihun bu tata..🙏
4,7/27/25,5:12 PM,Afri Kantin Baiiq MBR,insyaallah bisa bu.. bebas buat brp porsi.🙏


## Masking dan Pseudonimisasi Data Pribadi (PII Protection)

Penerapan regulasi perlindungan data pribadi (*Personally Identifiable Information* / PII) terhadap nomor telepon pelanggan:
- **Pelanggan**: Nomor kontak (`+62` atau string numerik) dipetakan ke identifier pseudonim bertahap (`Pelanggan_1`, `Pelanggan_2`, dst.) menggunakan tabel pemetaan (`kamus_anonim`) untuk memastikan konsistensi relasi di seluruh corpus percakapan.
- **Pengelola Kantin**: Identitas pihak kantin (`Afri Kantin Baiiq MBR`) dipertahankan tanpa perubahan untuk mempertahankan diferensiasi peran antara penyedia layanan dan konsumen dalam analisis transaksional.


In [13]:
# Inisialisasi lookup table dan counter pseudonim pelanggan
kamus_anonim = {}
id_pelanggan = 1

def sensor_nama(nama_asli):
    """
    Memetakan nomor kontak pelanggan ke identifier anonim unik.
    Identitas non-numerik (pengelola kantin) dipertahankan.
    """
    global id_pelanggan
    
    # Evaluasi apakah identitas pengirim merupakan nomor telepon
    if "+62" in nama_asli or nama_asli.replace(" ", "").replace("-", "").isdigit():
        if nama_asli not in kamus_anonim:
            kamus_anonim[nama_asli] = f"Pelanggan_{id_pelanggan}"
            id_pelanggan += 1
        return kamus_anonim[nama_asli]
    
    return nama_asli

# Transformasi kolom Pengirim menjadi Pengirim_Anonim
df['Pengirim_Anonim'] = df['Pengirim'].apply(sensor_nama)

# Verifikasi hasil pemetaan identitas
display(df[['Pengirim', 'Pengirim_Anonim', 'Pesan']].head(10))

,Pengirim,Pengirim_Anonim,Pesan
0,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR,<Media omitted>
1,+62 821-1146-0799,Pelanggan_1,nuhun infona bu 🙏🙏
2,+62 816-4801-781,Pelanggan_2,Walaikumsalam wrwb. Nasi tumpeng merah putih y...
3,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR,samihun bu tata..🙏
4,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR,insyaallah bisa bu.. bebas buat brp porsi.🙏
5,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR,yg ini untuk 20/25 porsi
6,+62 838-9283-5739,Pelanggan_3,Waalaikumussalam\nTrimakasih infonya bu🙏
7,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR,samihun bu endro🙏🤗
8,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR,assalamualaikum\nmaaf hari ini kantin libur du...
9,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR,Menu Kantin Baiiq\nSelasa 30.07.25\nGurame asa...


## Finalisasi Skema dan Serialisasi CSV

Tahap penutupan pemrosesan:
1. **Drop Kolom PII**: Kolom `Pengirim` asli dieliminasi dari memori untuk memastikan informasi nomor telepon tidak tersimpan pada media permanen.
2. **Serialisasi CSV**: Penyimpanan `DataFrame` ke berkas `dataset_wa.csv` dengan parameter `index=False` guna menghindari redundansi indeks numerik.


In [14]:
# Eliminasi kolom Pengirim yang memuat nomor kontak mentah
df = df.drop(['Pengirim'], axis=1)

# Ekspor DataFrame ke format CSV
df.to_csv(file_output, index=False)

print(f"Pemrosesan selesai: {len(df)} baris obrolan telah dibersihkan, disensor, dan disimpan ke '{file_output}'.")

Pemrosesan selesai: 13736 baris obrolan telah dibersihkan, disensor, dan disimpan ke 'dataset_wa.csv'.
